# Public Trade Analysis

Loads raw trades from `data/trades_raw/`, grouped by **wallet × market × timestamp**,
and enriches each trade with the **final token value** (winner / loser / price at resolution).

In [11]:
import json
import datetime
from pathlib import Path

import pandas as pd

## Parameters

In [12]:
# ── date range (inclusive) of market folders to load ──────────────────────────
START_DATE = datetime.date(2026, 1, 1)
END_DATE   = datetime.date(2026, 3, 6)

# ── path to trades_raw ────────────────────────────────────────────────────────
TRADES_RAW = Path("../data/trades_raw")

## 1 · Load markets and trades

In [13]:
import json
import datetime
from pathlib import Path

import pandas as pd

# try to use orjson for faster JSON parsing (install with: pip install orjson)
try:
    import orjson
    def json_loads(s):
        return orjson.loads(s)
except ImportError:
    def json_loads(s):
        return json.loads(s)

In [14]:
def load_market(json_path: Path) -> dict:
    """Load a single market definition from its .json file."""
    with json_path.open("rb") as f:
        return orjson.loads(f.read()) if "orjson" in dir() else json.load(f)


def load_trades(jsonl_path: Path) -> list[dict]:
    """Load all trades from a .jsonl file into a list of dicts."""
    rows = []
    with jsonl_path.open("rb") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json_loads(line)
            obj["asset"] = str(obj["asset"])
            rows.append(obj)
    return rows


def day_folders(root: Path, start: datetime.date, end: datetime.date) -> list[Path]:
    """Return sorted list of YYYY-MM-DD folders within [start, end]."""
    folders = []
    for p in sorted(root.iterdir()):
        if not p.is_dir():
            continue
        try:
            d = datetime.date.fromisoformat(p.name)
        except ValueError:
            continue
        if start <= d <= end:
            folders.append(p)
    return folders

In [15]:
folders = day_folders(TRADES_RAW, START_DATE, END_DATE)
print(f"Found {len(folders)} day-folder(s): {[f.name for f in folders]}")

Found 65 day-folder(s): ['2026-01-01', '2026-01-02', '2026-01-03', '2026-01-04', '2026-01-05', '2026-01-06', '2026-01-07', '2026-01-08', '2026-01-09', '2026-01-10', '2026-01-11', '2026-01-12', '2026-01-13', '2026-01-14', '2026-01-15', '2026-01-16', '2026-01-17', '2026-01-18', '2026-01-19', '2026-01-20', '2026-01-21', '2026-01-22', '2026-01-23', '2026-01-24', '2026-01-25', '2026-01-26', '2026-01-27', '2026-01-28', '2026-01-29', '2026-01-30', '2026-01-31', '2026-02-01', '2026-02-02', '2026-02-03', '2026-02-04', '2026-02-05', '2026-02-06', '2026-02-07', '2026-02-08', '2026-02-09', '2026-02-10', '2026-02-11', '2026-02-12', '2026-02-13', '2026-02-14', '2026-02-15', '2026-02-16', '2026-02-17', '2026-02-18', '2026-02-19', '2026-02-20', '2026-02-21', '2026-02-22', '2026-02-23', '2026-02-24', '2026-02-25', '2026-02-26', '2026-02-27', '2026-02-28', '2026-03-01', '2026-03-02', '2026-03-03', '2026-03-04', '2026-03-05', '2026-03-06']


In [16]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os

def load_market_and_trades(json_path: Path) -> tuple[dict, list[dict]] | None:
    """Load market definition and its trades from a single json/jsonl pair."""
    jsonl_path = json_path.with_suffix(".jsonl")
    if not jsonl_path.exists():
        return None
    
    market = load_market(json_path)
    trades = load_trades(jsonl_path)
    return market, trades


# collect all json file paths
all_json_paths = [
    json_path
    for folder in folders
    for json_path in sorted(folder.glob("*.json"))
]

all_trades: list[dict] = []
markets: dict[str, dict] = {}

num_workers = min(32, os.cpu_count() * 4 or 16)  # more threads for I/O-bound work

with ThreadPoolExecutor(max_workers=num_workers) as executor:
    futures = {executor.submit(load_market_and_trades, p): p for p in all_json_paths}
    
    for future in as_completed(futures):
        result = future.result()
        if result is None:
            continue
        
        market, trades = result
        condition_id = market["condition_id"]
        
        # keep only the first time we see a condition_id
        if condition_id not in markets:
            markets[condition_id] = market
        
        all_trades.extend(trades)

print(f"Loaded {len(all_trades):,} trade records across {len(markets):,} unique markets.")

Loaded 17,415,339 trade records across 98,799 unique markets.


## 2 · Build the trades DataFrame

In [18]:
df = pd.DataFrame(all_trades)

# parse timestamp → UTC datetime (vectorized)
df["dt"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

# canonical column names
df.rename(columns={"proxyWallet": "wallet", "conditionId": "condition_id"}, inplace=True)

# ensure numeric types (use pd.to_numeric for better performance)
df["size"] = pd.to_numeric(df["size"], errors="coerce")
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# sort chronologically (consider if you really need this - it's expensive)
df.sort_values(["condition_id", "wallet", "dt"], inplace=True, ignore_index=True)

print(df.shape)
df.head(3)

(17415339, 20)


,wallet,side,asset,condition_id,size,price,timestamp,title,slug,icon,eventSlug,outcome,outcomeIndex,name,pseudonym,bio,profileImage,profileImageOptimized,transactionHash,dt
0,0x1d3fd83676aabe718c13cba008e0a774b2127fec,BUY,2464494260851836388524244660612053848040667625...,0x00005c113846891686d99ff1d145ded8bdccfa028df0...,36.26,0.988447,1769905632,Will Jonathan Micallef win by KO or TKO?,ufc-oba-jon21-2026-01-31-micallef-win-by-ko-tko,https://polymarket-upload.s3.us-east-2.amazona...,ufc-oba-jon21-2026-01-31,No,1,Midrunner,Woozy-Copy,Inevitable,https://polymarket-upload.s3.us-east-2.amazona...,,0xbe77f57ad7cf076d224c718458709a7edc61b1e40a7f...,2026-02-01 00:27:12+00:00
1,0x5ee1c416fbe1c3e6a8bc51b74dd816c43276e116,BUY,2464494260851836388524244660612053848040667625...,0x00005c113846891686d99ff1d145ded8bdccfa028df0...,1287.00,0.999000,1769906342,Will Jonathan Micallef win by KO or TKO?,ufc-oba-jon21-2026-01-31-micallef-win-by-ko-tko,https://polymarket-upload.s3.us-east-2.amazona...,ufc-oba-jon21-2026-01-31,No,1,WuTangClan,Tempting-Catalogue,,,,0x4c82ec38eac8594af4ca7d8cedc668bf64796f2920bf...,2026-02-01 00:39:02+00:00
2,0x5ee1c416fbe1c3e6a8bc51b74dd816c43276e116,BUY,2464494260851836388524244660612053848040667625...,0x00005c113846891686d99ff1d145ded8bdccfa028df0...,706.00,0.999000,1769906428,Will Jonathan Micallef win by KO or TKO?,ufc-oba-jon21-2026-01-31-micallef-win-by-ko-tko,https://polymarket-upload.s3.us-east-2.amazona...,ufc-oba-jon21-2026-01-31,No,1,WuTangClan,Tempting-Catalogue,,,,0x05840ca9cf2976af584be395b84973e3bf2577585dd8...,2026-02-01 00:40:28+00:00


## 3 · Enrich with market metadata and final token value

For each trade we add:

| column | description |
|---|---|
| `question` | Human-readable market question |
| `end_date` | Market resolution date |
| `token_winner` | `True` if the traded token is the winning outcome |
| `final_price` | Final settlement price of the traded token (1.0 = winner, 0.0 = loser, or last known price if unresolved) |
| `trade_value_usdc` | `size × price` — USDC spent on the trade |
| `final_value_usdc` | `size × final_price` — USDC value of shares at resolution |

In [19]:
def build_token_lookup(markets: dict[str, dict]) -> dict[str, dict]:
    """Return {token_id → {condition_id, outcome, winner, final_price}} for all markets."""
    lookup: dict[str, dict] = {}
    for cid, m in markets.items():
        for tok in m.get("tokens", []):
            token_id = str(tok["token_id"])
            winner = bool(tok.get("winner", False))
            # if market is closed/resolved use binary settlement, else use last price
            if m.get("closed", False):
                final_price = 1.0 if winner else 0.0
            else:
                final_price = float(tok.get("price") or 0.0)
            lookup[token_id] = {
                "condition_id": cid,
                "outcome":      tok.get("outcome", ""),
                "token_winner": winner,
                "final_price":  final_price,
            }
    return lookup


token_lookup = build_token_lookup(markets)
print(f"Token lookup entries: {len(token_lookup):,}")

Token lookup entries: 197,598


In [20]:
# join token resolution data
token_df = pd.DataFrame.from_dict(token_lookup, orient="index")
token_df.index.name = "asset"
token_df.reset_index(inplace=True)

df = df.merge(
    token_df[["asset", "token_winner", "final_price"]],
    on="asset",
    how="left",
)

# join market-level metadata
market_meta = pd.DataFrame(
    [
        {
            "condition_id": cid,
            "question":     m["question"],
            "end_date":     pd.to_datetime(m["end_date_iso"], utc=True),
            "market_slug":  m["market_slug"],
        }
        for cid, m in markets.items()
    ]
)

df = df.merge(market_meta, on="condition_id", how="left")

# derived value columns
df["trade_value_usdc"] = df["size"] * df["price"]
df["final_value_usdc"] = df["size"] * df["final_price"]

print(df.shape)
df[["wallet", "condition_id", "dt", "side", "outcome", "size", "price",
    "token_winner", "final_price", "trade_value_usdc", "final_value_usdc"]].head(5)

(17415339, 27)


,wallet,condition_id,dt,side,outcome,size,price,token_winner,final_price,trade_value_usdc,final_value_usdc
0,0x1d3fd83676aabe718c13cba008e0a774b2127fec,0x00005c113846891686d99ff1d145ded8bdccfa028df0...,2026-02-01 00:27:12+00:00,BUY,No,36.260000,0.988447,True,1.0,35.841100,36.26
1,0x5ee1c416fbe1c3e6a8bc51b74dd816c43276e116,0x00005c113846891686d99ff1d145ded8bdccfa028df0...,2026-02-01 00:39:02+00:00,BUY,No,1287.000000,0.999000,True,1.0,1285.713000,1287.00
2,0x5ee1c416fbe1c3e6a8bc51b74dd816c43276e116,0x00005c113846891686d99ff1d145ded8bdccfa028df0...,2026-02-01 00:40:28+00:00,BUY,No,706.000000,0.999000,True,1.0,705.294000,706.00
3,0x5ee1c416fbe1c3e6a8bc51b74dd816c43276e116,0x00005c113846891686d99ff1d145ded8bdccfa028df0...,2026-02-01 00:43:00+00:00,BUY,No,853.000000,0.999000,True,1.0,852.147000,853.00
4,0xc30029f4dc58619bb58ab4b6be3532eb9cf025a0,0x00005c113846891686d99ff1d145ded8bdccfa028df0...,2026-01-29 13:53:22+00:00,BUY,Yes,7.285713,0.280000,False,0.0,2.039999,0.00


## 4 · Group by wallet × market × timestamp

Each row in `grouped` represents all trades made by one wallet in one market at
one exact timestamp (i.e. a single on-chain transaction can place multiple fills).

In [21]:
GROUP_KEYS = ["wallet", "condition_id", "dt"]

grouped = (
    df.groupby(GROUP_KEYS, sort=False)
    .agg(
        question          = ("question",          "first"),
        market_slug       = ("market_slug",        "first"),
        end_date          = ("end_date",           "first"),
        side              = ("side",               "first"),   # BUY / SELL
        outcome           = ("outcome",            "first"),   # Yes / No
        token_winner      = ("token_winner",       "first"),
        final_price       = ("final_price",        "first"),
        # sum across fills in the same tx
        total_size        = ("size",               "sum"),
        avg_price         = ("price",              "mean"),
        trade_value_usdc  = ("trade_value_usdc",   "sum"),
        final_value_usdc  = ("final_value_usdc",   "sum"),
        num_fills         = ("transactionHash",    "count"),
    )
    .reset_index()
    .sort_values(["wallet", "condition_id", "dt"])
    .reset_index(drop=True)
)

print(f"Grouped rows: {len(grouped):,}  (from {len(df):,} raw trades)")
grouped.head(10)

Grouped rows: 16,356,433  (from 17,415,339 raw trades)


,wallet,condition_id,dt,question,market_slug,end_date,side,outcome,token_winner,final_price,total_size,avg_price,trade_value_usdc,final_value_usdc,num_fills
0,0x0000075afc60ebc564d56be0bc23eaceea67d446,0x2ece3af7241e819906ef9f99cc850ba7feaa7f7dc41a...,2026-01-09 07:47:14+00:00,Will Tesla reach $645 in January?,will-tsla-reach-645-in-january,2026-02-01 00:00:00+00:00,BUY,No,True,1.0,13.052000,0.996,12.999792,13.052000,1
1,0x0000075afc60ebc564d56be0bc23eaceea67d446,0x79d7663a5a4b32512cbb86ce4b1a465d0bee5ec544e2...,2026-02-10 08:03:07+00:00,Will the Logan Paul 1st Edition Charizard sale...,will-the-logan-paul-1st-edition-charizard-sale...,2026-02-28 00:00:00+00:00,BUY,Yes,True,1.0,0.700000,0.990,0.693000,0.700000,1
2,0x0000075afc60ebc564d56be0bc23eaceea67d446,0x79d7663a5a4b32512cbb86ce4b1a465d0bee5ec544e2...,2026-02-10 08:03:35+00:00,Will the Logan Paul 1st Edition Charizard sale...,will-the-logan-paul-1st-edition-charizard-sale...,2026-02-28 00:00:00+00:00,BUY,Yes,True,1.0,15.200000,0.991,15.063200,15.200000,1
3,0x0000075afc60ebc564d56be0bc23eaceea67d446,0xe9ec9b9d878c85fd70038f711c85b81a576e4806708a...,2026-01-28 08:22:37+00:00,Will NVIDIA be the largest company in the worl...,will-nvidia-be-the-largest-company-in-the-worl...,2026-01-31 00:00:00+00:00,BUY,Yes,True,1.0,19.000000,0.996,18.924000,19.000000,1
4,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,0x212542b6d0eb29dce20144d0aa9e3a024e83c6c4340f...,2026-01-01 16:09:31+00:00,Golden Knights vs. Blues,nhl-las-stl-2026-01-02,2026-01-02 00:00:00+00:00,BUY,Blues,True,1.0,7.000000,0.430,3.010000,7.000000,1
5,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,0x231f643e4aa7846eff5565b3ba1a197cc27d4c7e8a53...,2026-01-19 21:27:02+00:00,Devils vs. Oilers,nhl-nj-edm-2026-01-20,2026-01-21 00:00:00+00:00,BUY,Devils,True,1.0,5.000000,0.410,2.050000,5.000000,1
6,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,0x3e31916dc44ff55fae016f75e61566be3e77c094d3f8...,2026-02-20 17:30:58+00:00,Will China win the second most gold medals in ...,will-china-win-the-second-most-gold-medals-in-...,2026-02-22 00:00:00+00:00,BUY,Yes,False,0.0,16643.640000,0.001,16.643640,0.000000,1
7,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,0x6a37acdfd255152d885b5d9655d133e1775243599825...,2026-01-05 06:11:43+00:00,Will the highest temperature in New York City ...,highest-temperature-in-nyc-on-january-4-38-39f,2026-01-04 00:00:00+00:00,BUY,Yes,False,0.0,1000.000000,0.001,1.000000,0.000000,1
8,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,0x6ed5173182f270baafbb61c3ca07eb709462f9eaa654...,2026-01-03 09:41:21+00:00,Golden Knights vs. Blackhawks,nhl-las-chi-2026-01-04,2026-01-05 00:00:00+00:00,BUY,Blackhawks,True,1.0,9.729728,0.370,3.599999,9.729728,1
9,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,0x7e37855e9e6bd88c37ebb0e2369a93db2472a7ae2b53...,2026-02-13 04:44:39+00:00,Valorant: DetonatioN FocusMe vs T1 (BO3) - VCT...,val-dfm1-t1-2026-02-13,2026-02-13 00:00:00+00:00,BUY,DetonatioN FocusMe,False,0.0,5.000000,0.310,1.550000,0.000000,1


## 5 · Wallet-level P&L summary

Aggregate per wallet across all markets: total USDC spent, total final value, and net P&L.

In [22]:
# sign convention: BUY spends USDC, SELL receives USDC
grouped["signed_cost"] = grouped.apply(
    lambda r: r["trade_value_usdc"] if r["side"] == "BUY" else -r["trade_value_usdc"],
    axis=1,
)
grouped["signed_final"] = grouped.apply(
    lambda r: r["final_value_usdc"] if r["side"] == "BUY" else -r["final_value_usdc"],
    axis=1,
)

wallet_summary = (
    grouped.groupby("wallet")
    .agg(
        num_markets       = ("condition_id",  "nunique"),
        num_trades        = ("num_fills",      "sum"),
        total_cost_usdc   = ("signed_cost",    "sum"),
        total_final_usdc  = ("signed_final",   "sum"),
    )
    .assign(pnl_usdc=lambda x: x["total_final_usdc"] - x["total_cost_usdc"])
    .sort_values("pnl_usdc", ascending=False)
    .reset_index()
)

print(f"Unique wallets: {len(wallet_summary):,}")
wallet_summary.head(20)

Unique wallets: 424,873


,wallet,num_markets,num_trades,total_cost_usdc,total_final_usdc,pnl_usdc
0,0xac44cb78be973ec7d91b69678c4bdfa7009afbd7,8,93,5.148077e+06,7.332569e+06,2.184492e+06
1,0xd25c72ac0928385610611c8148803dc717334d20,6,110,2.170337e+06,3.940643e+06,1.770305e+06
2,0xd0b4c4c020abdc88ad9a884f999f3d8cff8ffed6,459,1255,1.357415e+07,1.526077e+07,1.686625e+06
3,0x1af1dfc2c523af1d7551597c985277cd11b30f7b,24,360,-3.154208e+06,-1.770606e+06,1.383601e+06
4,0x91654fd592ea5339fc0b1b2f2b30bfffa5e75b98,132,839,1.087988e+07,1.204534e+07,1.165460e+06
5,0xd82079c0d6b837bad90abf202befc079da5819f6,9,198,2.931503e+06,3.937196e+06,1.005693e+06
6,0x204f72f35326db932158cba6adff0b9a1da95e14,20739,535373,8.072305e+07,8.166932e+07,9.462702e+05
7,0xdb27bf2ac5d428a9c63dbc914611036855a6c56e,104,155,2.495124e+06,3.431452e+06,9.363280e+05
8,0x9c16127eccf031df45461ef1e04b52ea286a09cb,67,85,1.200248e+06,1.977836e+06,7.775873e+05
9,0xc2e7800b5af46e6093872b177b7a5e7f0563be51,85,695,7.498805e+07,7.575186e+07,7.638029e+05


## 6 · Market-level volume summary

In [23]:
market_summary = (
    grouped.groupby(["condition_id", "question", "end_date", "token_winner"])
    .agg(
        num_wallets      = ("wallet",            "nunique"),
        num_trades       = ("num_fills",          "sum"),
        volume_usdc      = ("trade_value_usdc",   "sum"),
        final_value_usdc = ("final_value_usdc",   "sum"),
    )
    .reset_index()
    .sort_values("volume_usdc", ascending=False)
    .reset_index(drop=True)
)

print(f"Unique markets: {len(market_summary):,}")
market_summary.head(10)

Unique markets: 161,671


,condition_id,question,end_date,token_winner,num_wallets,num_trades,volume_usdc,final_value_usdc
0,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3...,Khamenei out as Supreme Leader of Iran by Febr...,2026-02-28 00:00:00+00:00,True,652,1348,5.984009e+06,5.991159e+06
1,0x43ec78527bd98a0588dd9455685b2cc82f5743140cb3...,US government shutdown Saturday?,2026-01-31 00:00:00+00:00,True,698,1771,5.560002e+06,5.568061e+06
2,0xe0e9b323fd61665061deb2b2a0b91668eb4222fc698e...,Will Liverpool FC win on 2026-01-31?,2026-01-31 00:00:00+00:00,True,865,1982,5.456335e+06,8.910043e+06
3,0x90cda6e854383eff1f0599306b70341775771d32168e...,Will Paris Saint-Germain FC win on 2026-01-28?,2026-01-28 00:00:00+00:00,False,849,1714,4.959035e+06,8.866000e+01
4,0x05beef793157deb1cc34e2d3159539f461b73c7eeaa3...,U.S. anti-cartel ground operation in Mexico by...,2026-01-31 00:00:00+00:00,True,541,1800,4.480937e+06,4.493220e+06
5,0xdece3c9a32a0244f9dc4622feadd31f74dec86b02399...,Will Villarreal CF win on 2026-01-20?,2026-01-20 00:00:00+00:00,False,448,1050,3.964527e+06,9.000000e-02
6,0xab4fab2240e0ff00691add26fa2c5d56d715dbbc1226...,Will Liverpool FC win on 2026-01-21?,2026-01-21 00:00:00+00:00,True,745,1650,3.781711e+06,6.918976e+06
7,0x5738020f4942e54c0124f8b3576d028e383484d26d3f...,Will Paris Saint-Germain FC win on 2026-01-20?,2026-01-20 00:00:00+00:00,False,993,2083,3.765606e+06,8.491893e+00
8,0x7b6629cca58b0a3be21a2e448f8d92c930e78d26d5f6...,Will Tottenham Hotspur FC win on 2026-01-17?,2026-01-17 00:00:00+00:00,True,311,981,3.752346e+06,7.443584e+06
9,0x41e47408f8ab39b46a9d9e3c9b15ebd62f1d795eb072...,"US x Iran meeting by February 6, 2026?",2026-02-13 00:00:00+00:00,True,487,2086,3.655457e+06,3.660281e+06


## 7 · Wallet statistics (quantiles)

Distribution of activity and P&L across wallets.

In [24]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
N = len(wallet_summary)

wallet_stats = (
    wallet_summary[["num_trades", "num_markets", "pnl_usdc"]]
    .quantile(QUANTILES)
    .rename_axis("quantile")
)

# number of wallets at or below each quantile threshold
wallet_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])

# append count, mean and std for reference
wallet_stats.loc["count"] = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].count()
wallet_stats.loc["mean"]  = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].mean()
wallet_stats.loc["std"]   = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].std()
wallet_stats.loc["count", "wallet_count"] = N
wallet_stats.loc["mean",  "wallet_count"] = float("nan")
wallet_stats.loc["std",   "wallet_count"] = float("nan")

wallet_stats["wallet_count"] = wallet_stats["wallet_count"].astype("Int64")
wallet_stats["num_trades"]   = wallet_stats["num_trades"].round(1)
wallet_stats["num_markets"]  = wallet_stats["num_markets"].round(1)
wallet_stats["pnl_usdc"]     = wallet_stats["pnl_usdc"].round(2)

wallet_stats

,wallet_count,num_trades,num_markets,pnl_usdc
quantile,,,,
0.001,425,1.0,1.0,-21773.26
0.01,4249,1.0,1.0,-2196.72
0.05,21244,1.0,1.0,-352.90
0.1,42487,1.0,1.0,-105.43
0.25,106218,1.0,1.0,-14.47
0.5,212436,3.0,2.0,-0.10
0.75,318655,10.0,7.0,1.76
0.9,382386,35.0,22.0,36.75
0.95,403629,85.0,46.0,176.10


## 8 · Full enriched trade table

The main output: one row per wallet × market × timestamp, with all enrichment columns.

In [25]:
DISPLAY_COLS = [
    "wallet", "condition_id", "dt", "end_date",
    "question", "side", "outcome", "total_size", "avg_price",
    "token_winner", "final_price",
    "trade_value_usdc", "final_value_usdc", "num_fills",
]

grouped[DISPLAY_COLS]

,wallet,condition_id,dt,end_date,question,side,outcome,total_size,avg_price,token_winner,final_price,trade_value_usdc,final_value_usdc,num_fills
0,0x0000075afc60ebc564d56be0bc23eaceea67d446,0x2ece3af7241e819906ef9f99cc850ba7feaa7f7dc41a...,2026-01-09 07:47:14+00:00,2026-02-01 00:00:00+00:00,Will Tesla reach $645 in January?,BUY,No,13.052,0.996000,True,1.0,12.999792,13.052,1
1,0x0000075afc60ebc564d56be0bc23eaceea67d446,0x79d7663a5a4b32512cbb86ce4b1a465d0bee5ec544e2...,2026-02-10 08:03:07+00:00,2026-02-28 00:00:00+00:00,Will the Logan Paul 1st Edition Charizard sale...,BUY,Yes,0.700,0.990000,True,1.0,0.693000,0.700,1
2,0x0000075afc60ebc564d56be0bc23eaceea67d446,0x79d7663a5a4b32512cbb86ce4b1a465d0bee5ec544e2...,2026-02-10 08:03:35+00:00,2026-02-28 00:00:00+00:00,Will the Logan Paul 1st Edition Charizard sale...,BUY,Yes,15.200,0.991000,True,1.0,15.063200,15.200,1
3,0x0000075afc60ebc564d56be0bc23eaceea67d446,0xe9ec9b9d878c85fd70038f711c85b81a576e4806708a...,2026-01-28 08:22:37+00:00,2026-01-31 00:00:00+00:00,Will NVIDIA be the largest company in the worl...,BUY,Yes,19.000,0.996000,True,1.0,18.924000,19.000,1
4,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,0x212542b6d0eb29dce20144d0aa9e3a024e83c6c4340f...,2026-01-01 16:09:31+00:00,2026-01-02 00:00:00+00:00,Golden Knights vs. Blues,BUY,Blues,7.000,0.430000,True,1.0,3.010000,7.000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16356428,0xffffffe1e093aacd21e4e281e66d543fb0b23455,0xb4bc113c6da82db67b6b17d39369de522e61425dd360...,2026-02-03 22:00:06+00:00,2026-02-03 00:00:00+00:00,"US strikes Iran by February 3, 2026?",BUY,Yes,6000.000,0.005000,False,0.0,30.000000,0.000,1
16356429,0xffffffe1e093aacd21e4e281e66d543fb0b23455,0xb4bc113c6da82db67b6b17d39369de522e61425dd360...,2026-02-03 22:21:18+00:00,2026-02-03 00:00:00+00:00,"US strikes Iran by February 3, 2026?",BUY,Yes,6000.000,0.005000,False,0.0,30.000000,0.000,1
16356430,0xffffffe1e093aacd21e4e281e66d543fb0b23455,0xb4bc113c6da82db67b6b17d39369de522e61425dd360...,2026-02-03 22:46:16+00:00,2026-02-03 00:00:00+00:00,"US strikes Iran by February 3, 2026?",BUY,Yes,3011.725,0.005977,False,0.0,18.000000,0.000,1
16356431,0xffffffe1e093aacd21e4e281e66d543fb0b23455,0xb4bc113c6da82db67b6b17d39369de522e61425dd360...,2026-02-03 23:00:30+00:00,2026-02-03 00:00:00+00:00,"US strikes Iran by February 3, 2026?",BUY,Yes,3000.000,0.005000,False,0.0,15.000000,0.000,1


## 9 · Export top-5% wallets by P&L

Writes trades for the top 5% of wallets (by `pnl_usdc`) to `data/trades_processed/`,
mirroring the `trades_raw/` structure: one folder per market end-date,
`<condition_id_prefix>.json` (market definition) and `<condition_id_prefix>.jsonl` (trades).  
Trades are individual fills (not grouped), enriched with `trade_value_usdc` and `final_value_usdc`.

In [26]:
# calculate pnl of a specific wallet, when side=BUY cost is negative, when side=SELL cost is positive

df[df["wallet"] == "0xe51abbc214c832cf44463db0f5f3ed421c742665"].apply(lambda r: -r["trade_value_usdc"] if r["side"] == "BUY" else r["trade_value_usdc"], axis=1).sum()

np.float64(-92.66005900000005)

In [27]:
import shutil

TRADES_PROCESSED = Path("../data/trades_processed")

# ── identify top-5% wallets by P&L ───────────────────────────────────────────
pnl_threshold = wallet_summary["pnl_usdc"].quantile(0.95)
top_wallets   = set(wallet_summary.loc[wallet_summary["pnl_usdc"] >= pnl_threshold, "wallet"])
print(f"P&L threshold (95th pct): {pnl_threshold:,.2f} USDC")
print(f"Top-5% wallet count:      {len(top_wallets):,}")

# ── filter raw individual trades (df) to those wallets ───────────────────────
top_trades = df[df["wallet"].isin(top_wallets)].copy()
print(f"Trade rows to export:     {len(top_trades):,}")

# print deciles of P&L for top wallets
for i in range(0, 11):
    q = i / 10
    pnl_q = wallet_summary.loc[wallet_summary["pnl_usdc"] >= pnl_threshold, "pnl_usdc"].quantile(q)
    print(f"  P&L at {q:.0%} pct: {pnl_q:,.2f} USDC")

P&L threshold (95th pct): 176.10 USDC
Top-5% wallet count:      21,244
Trade rows to export:     5,740,685
  P&L at 0% pct: 176.10 USDC
  P&L at 10% pct: 217.52 USDC
  P&L at 20% pct: 278.39 USDC
  P&L at 30% pct: 363.59 USDC
  P&L at 40% pct: 481.18 USDC
  P&L at 50% pct: 646.38 USDC
  P&L at 60% pct: 862.08 USDC
  P&L at 70% pct: 1,207.51 USDC
  P&L at 80% pct: 1,744.05 USDC
  P&L at 90% pct: 3,350.23 USDC
  P&L at 100% pct: 2,184,492.18 USDC


In [28]:
EXPORT_COLS = [
    "wallet", "condition_id", "dt",
    "side", "asset", "outcome", "outcomeIndex",
    "size", "price",
    "trade_value_usdc", "final_value_usdc",
    "token_winner", "final_price",
    "name", "pseudonym", "transactionHash", 'title'
]

# wipe previous output so re-runs are idempotent
if TRADES_PROCESSED.exists():
    shutil.rmtree(TRADES_PROCESSED)

written_jsonl  = 0
written_json   = 0
written_rows   = 0

# group by market end-date → condition_id (mirrors trades_raw layout)
for (end_date, condition_id), market_trades in top_trades.groupby(
    [top_trades["end_date"].dt.date, "condition_id"],
    sort=False,
):
    day_dir = TRADES_PROCESSED / str(end_date)
    day_dir.mkdir(parents=True, exist_ok=True)

    # same 14-char hex prefix convention as trades_raw: '0x' + 14 hex chars
    file_stem = condition_id[:16]

    # ── market definition .json ───────────────────────────────────────────────
    json_path = day_dir / f"{file_stem}.json"
    if not json_path.exists():
        with json_path.open("w") as f:
            json.dump(markets[condition_id], f)
        written_json += 1

    # ── individual trade fills .jsonl ─────────────────────────────────────────
    rows = market_trades[EXPORT_COLS].copy()
    rows["dt"] = rows["dt"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

    jsonl_path = day_dir / f"{file_stem}.jsonl"
    with jsonl_path.open("w") as f:
        for record in rows.to_dict(orient="records"):
            f.write(json.dumps(record) + "\n")

    written_jsonl += 1
    written_rows  += len(rows)

print(f"Written {written_rows:,} trade rows  → {written_jsonl:,} .jsonl files")
print(f"Written {written_json:,} market definitions → .json files")
print(f"Output: {TRADES_PROCESSED}")

Written 5,740,685 trade rows  → 91,427 .jsonl files
Written 91,427 market definitions → .json files
Output: ../data/trades_processed
